# RT Notebook 18
## Domain Projection and Primitive Hierarchy

### Central question

> How can the single primitive relational condition `(*|*)` remain invariant while increasingly expressive structures become admissible as domain degrees of freedom are gained?

### Scope

The black-hole or singularity comparison is an **analogy only**. This notebook does not identify `(*|*)` with an astrophysical object.

### Working hypotheses

1. `(*|*)` is the unique primitive in the core relational domain.
2. Primitive status is indexed by domain.
3. A non-primitive concept may remain meaningful in a domain.
4. Gaining DoF enlarges admissible ordinal organization.
5. Projection may preserve invariants while leaving residue.
6. Resolution can be both an inferential entry condition and a closure condition without implying temporal beginning or end.
7. Ordinal orientation is not identical to temporal order.

## Interpretation protocol

- **Core primitive**: irreducible in the core domain.
- **Local primitive**: treated as primitive within a declared dominant domain.
- **Projected object**: represented in a target domain by an explicit map.
- **Derived object**: constructed using admitted relations in a domain.
- **Meaningful aspect**: interpretable although not primitive.
- **DoF**: independently admissible orientation or organizational capacity.
- **Resolution**: domain-conditioned realization or closure.
- **Residue**: relational information not preserved by projection.

Every interpretation must declare its domain first.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from itertools import product
from typing import Any, Callable, Dict, FrozenSet, Iterable, Mapping, Optional, Tuple
import math, json, hashlib
print("RT Notebook 18 initialized")

## 1. Formal vocabulary

In [ ]:
class Status(str, Enum):
    CORE_PRIMITIVE = "core_primitive"
    LOCAL_PRIMITIVE = "local_primitive"
    DERIVED = "derived"
    PROJECTED = "projected"
    EVALUATED = "evaluated"
    CONSTRAINED = "constrained"
    MEANINGFUL_ASPECT = "meaningful_aspect"

@dataclass(frozen=True)
class Domain:
    name: str
    dof: int
    dominant_concepts: FrozenSet[str] = frozenset()
    admissible_relations: FrozenSet[str] = frozenset()
    description: str = ""
    def __post_init__(self):
        if self.dof < 0:
            raise ValueError("DoF must be non-negative")

@dataclass(frozen=True)
class PrimitiveCondition:
    symbol: str = "(*|*)"
    left_aspect: str = "*"
    distinction: str = "|"
    right_aspect: str = "*"
    def signature(self):
        return (self.left_aspect, self.distinction, self.right_aspect)

@dataclass(frozen=True)
class DomainObject:
    name: str
    domain: str
    status: Status
    representation: Any
    provenance: Tuple[str, ...] = ()
    metadata: Mapping[str, Any] = field(default_factory=dict)

primitive = PrimitiveCondition()
CORE = Domain("core_relational", 0, frozenset({"distinction_condition"}), frozenset({"distinguish","couple","exclude"}))
ORDINAL = Domain("ordinal_orientation", 1, frozenset({"orientation"}), frozenset({"left_of","right_of","reference_to"}))
GRADIENT = Domain("gradient_array", 2, frozenset({"gradient","polarity","reference"}), frozenset({"opposes","balances","deviates"}))
TEMPORAL = Domain("temporal", 3, frozenset({"time","duration","before","after"}), frozenset({"precedes","follows","persists"}))
STATISTICAL = Domain("statistical", 3, frozenset({"frequency","distribution","expectation"}), frozenset({"samples","aggregates","estimates"}))
DOMAINS={d.name:d for d in [CORE,ORDINAL,GRADIENT,TEMPORAL,STATISTICAL]}
print("Primitive:", primitive.signature())
print("Domains:", list(DOMAINS))

## 2. Domain validation

In [ ]:
def validate_domain_object(obj: DomainObject):
    if obj.domain not in DOMAINS:
        raise ValueError(f"Unknown domain: {obj.domain}")
    if obj.status == Status.CORE_PRIMITIVE and obj.domain != CORE.name:
        raise ValueError("CORE_PRIMITIVE is only admissible in the core domain")
    if obj.name == primitive.symbol and obj.domain == CORE.name and obj.status != Status.CORE_PRIMITIVE:
        raise ValueError("(*|*) must be core primitive in the core domain")

objects = [
    DomainObject(primitive.symbol, CORE.name, Status.CORE_PRIMITIVE, primitive.signature()),
    DomainObject("time", TEMPORAL.name, Status.LOCAL_PRIMITIVE, "t"),
    DomainObject("time", CORE.name, Status.PROJECTED, "temporal aspect", (primitive.symbol,)),
]
for obj in objects:
    validate_domain_object(obj)
print("Domain declarations validated")

### Governing distinction

\[

eg\operatorname{Primitive}(x,D)\;
ot\Rightarrow\;
eg\operatorname{Meaningful}(x,D)
\]

Time can therefore be locally primitive in a temporal domain while temporal aspects remain meaningful, but projected or derived, in another domain.

## 3. Domain-relative status table

In [ ]:
STATUS_TABLE = {
    CORE.name: {primitive.symbol:Status.CORE_PRIMITIVE, "ordinal_orientation":Status.DERIVED, "time":Status.PROJECTED, "resolution":Status.MEANINGFUL_ASPECT},
    ORDINAL.name: {primitive.symbol:Status.PROJECTED, "ordinal_orientation":Status.LOCAL_PRIMITIVE, "time":Status.MEANINGFUL_ASPECT, "resolution":Status.DERIVED},
    GRADIENT.name: {primitive.symbol:Status.PROJECTED, "gradient":Status.LOCAL_PRIMITIVE, "time":Status.MEANINGFUL_ASPECT, "resolution":Status.DERIVED},
    TEMPORAL.name: {primitive.symbol:Status.PROJECTED, "ordinal_orientation":Status.DERIVED, "time":Status.LOCAL_PRIMITIVE, "resolution":Status.DERIVED},
}
for domain, entries in STATUS_TABLE.items():
    print(f"\n[{domain}]")
    for concept,status in entries.items():
        print(f"  {concept:22s} -> {status.value}")

## 4. DoF as admissible organizational capacity

In [ ]:
def relation_capacity(dof:int, relation_types:int=1)->int:
    return 0 if dof < 2 else math.comb(dof,2)*relation_types

def binary_orientations(dof:int):
    return [tuple()] if dof==0 else list(product((-1,1), repeat=dof))

for n in range(7):
    print(f"DoF={n}: orientations={len(binary_orientations(n)):3d}, pair capacity={relation_capacity(n):2d}, typed capacity={relation_capacity(n,3):2d}")

This operationalizes gained DoF as enlarged relational expressibility, not as elapsed time. It does not yet prove that DoF is ontologically generated by projection.

## 5. Projection operators and residue

In [ ]:
@dataclass(frozen=True)
class RelationalState:
    label: str
    orientations: Tuple[int,...]
    relations: FrozenSet[Tuple[int,int,str]]
    source_signature: Tuple[str,str,str] = primitive.signature()
    @property
    def dof(self): return len(self.orientations)
    def canonical(self): return (self.source_signature, self.orientations, tuple(sorted(self.relations)))

@dataclass(frozen=True)
class ProjectionResult:
    source_domain: str
    target_domain: str
    source: Any
    image: Any
    preserved: FrozenSet[str]
    residue: Any
    operator_name: str

@dataclass(frozen=True)
class ProjectionOperator:
    name: str
    source_domain: str
    target_domain: str
    map_fn: Callable[[Any],Any]
    residue_fn: Callable[[Any,Any],Any]
    preserved_fn: Callable[[Any,Any],Iterable[str]]
    def apply(self, source):
        image=self.map_fn(source)
        return ProjectionResult(self.source_domain,self.target_domain,source,image,frozenset(self.preserved_fn(source,image)),self.residue_fn(source,image),self.name)

def p_to_o(p):
    return RelationalState("first_orientation",(-1,1),frozenset({(0,1,"distinguished_from")}),p.signature())
def residue(p,img):
    return {"unresolved_aspect_identity":(p.left_aspect,p.right_aspect),"projection_choice":"left/right imposed"}
def preserved(p,img):
    yield "primitive_signature"
    yield "distinction"

PI_CORE_ORDINAL=ProjectionOperator("Pi_core_to_ordinal",CORE.name,ORDINAL.name,p_to_o,residue,preserved)
r=PI_CORE_ORDINAL.apply(primitive)
print("Image:",r.image)
print("Preserved:",sorted(r.preserved))
print("Residue:",r.residue)

Projection is represented as

\[
\Pi_{D_s	o D_t}(x)=(x',R)
\]

where residue is relational information not preserved by the map. It is not automatically energy, noise, or error.

## 6. Non-unique projection from one primitive

In [ ]:
def left_map(p):
    return RelationalState("left_reference",(0,1),frozenset({(0,1,"reference_to")}),p.signature())
def right_map(p):
    return RelationalState("right_reference",(-1,0),frozenset({(1,0,"reference_to")}),p.signature())
def generic_residue(s,i): return {"image":i.label,"nonuniqueness":True}
def generic_preserved(s,i): return ("primitive_signature","distinction")
PI_LEFT=ProjectionOperator("Pi_left",CORE.name,ORDINAL.name,left_map,generic_residue,generic_preserved)
PI_RIGHT=ProjectionOperator("Pi_right",CORE.name,ORDINAL.name,right_map,generic_residue,generic_preserved)
left=PI_LEFT.apply(primitive); right=PI_RIGHT.apply(primitive)
print("Same source:",left.source==right.source)
print("Same image:",left.image.canonical()==right.image.canonical())
print(left.image.canonical())
print(right.image.canonical())

The executable example supports the provisional form

\[
((*|*),\Pi,ho,D_t)\Rightarrow x'
\]

rather than assuming that `(*|*)` alone uniquely determines one higher-domain organization.

## 7. Resolution is many-to-one

In [ ]:
@dataclass(frozen=True)
class Resolution:
    value: Any
    domain: str
    rule: str
    witness: Tuple[Any,...]

def signed_balance(state,domain):
    return Resolution(sum(state.orientations),domain,"signed_balance",state.canonical())
def density_resolution(state,domain):
    return Resolution(sum(abs(x) for x in state.orientations),domain,"absolute_density",state.canonical())

examples=[
    RelationalState("A",(1,1,1,1),frozenset()),
    RelationalState("B",(-1,-1,1,1),frozenset()),
    RelationalState("C",(-1,1,-1,1),frozenset()),
]
for s in examples:
    print(s.label, signed_balance(s,GRADIENT.name).value, density_resolution(s,GRADIENT.name).value)

Distinct organizations can share the same resolution. Resolution therefore supplies a closure condition but not a unique organization identity.

It can be traversed in either inferential orientation:

\[
O\mapsto \mathcal R_D(O)
\]

or

\[
r\mapsto\{O:\mathcal R_D(O)=r\}.
\]

This is ordinal reversibility of inquiry, not temporal reversal.

## 8. Resolution-conditioned reconstruction

In [ ]:
def matching_organizations(dof,target):
    out=[]
    for i,o in enumerate(binary_orientations(dof)):
        s=RelationalState(f"s{i}",o,frozenset())
        if signed_balance(s,GRADIENT.name).value==target:
            out.append(s)
    return out
matching=matching_organizations(4,0)
print("Number resolving to 0:",len(matching))
for s in matching: print(s.orientations)

## 9. Projection path dependence

In [ ]:
def ordinal_to_gradient(state):
    o=state.orientations+(0,)
    rel=set(state.relations)
    rel.update({(0,2,"deviates"),(1,2,"deviates")})
    return RelationalState(state.label+"_gradient",o,frozenset(rel),state.source_signature)
def direct_core_gradient(p):
    return RelationalState("direct_gradient",(0,1,-1),frozenset({(0,1,"reference_to"),(1,2,"opposes")}),p.signature())
path_a=ordinal_to_gradient(left.image)
path_b=direct_core_gradient(primitive)
print("Sequential:",path_a.canonical())
print("Direct:",path_b.canonical())
print("Path independent:",path_a.canonical()==path_b.canonical())

This constructed counterexample shows that path independence cannot be assumed:

\[
\Pi_{O	o G}\circ\Pi_{C	o O}
e\Pi_{C	o G}.
\]

Notebook 19 should determine the conditions under which equality does hold.

## 10. Meaningful aspects of time outside the temporal domain

In [ ]:
@dataclass(frozen=True)
class TemporalAspect:
    name: str
    requires_time_primitive: bool
    ordinal_surrogate: Optional[str]
    interpretation: str

aspects=[
    TemporalAspect("before_after",False,"precedence relation","Representable as ordinal orientation without duration"),
    TemporalAspect("duration",True,None,"Requires metric temporal structure"),
    TemporalAspect("persistence",False,"invariance across ordered evaluations","Can remain meaningful without core time"),
    TemporalAspect("rate",True,"change per ordinal update only if scale is declared","Requires a denominator and metric"),
]
for a in aspects:
    print(f"{a.name:14s} | requires time primitive={a.requires_time_primitive} | surrogate={a.ordinal_surrogate}")

## 11. Automated consistency checks

In [ ]:
def primitive_preserved(state): return state.source_signature==primitive.signature()
checks={
    "core_primitive_valid": objects[0].status==Status.CORE_PRIMITIVE,
    "time_local_primitive_temporal": objects[1].status==Status.LOCAL_PRIMITIVE,
    "time_projected_core": objects[2].status==Status.PROJECTED,
    "projection_nonunique": left.image.canonical()!=right.image.canonical(),
    "resolution_many_to_one": len(matching)>1,
    "path_dependence_example": path_a.canonical()!=path_b.canonical(),
    "primitive_signature_preserved": all(primitive_preserved(s) for s in [r.image,left.image,right.image,path_a,path_b]),
}
for k,v in checks.items(): print(f"{k:38s}: {'PASS' if v else 'FAIL'}")
assert all(checks.values())
print("All checks passed")

## 12. Falsification targets

In [ ]:
TESTS=[
 {"id":"T18-01","question":"Does added DoF strictly enlarge admissible organization?","failure":"No new inequivalent organizations appear."},
 {"id":"T18-02","question":"Is projection unique under fixed domain, operator, and reference?","failure":"Inequivalent images remain under identical declarations."},
 {"id":"T18-03","question":"Is any nontrivial invariant preserved across all admissible projections?","failure":"Only bookkeeping labels survive."},
 {"id":"T18-04","question":"When is projection composition path independent?","failure":"Direct and composed projections remain systematically inequivalent."},
 {"id":"T18-05","question":"Can resolution uniquely recover organization?","failure":"Distinct organizations share resolution."},
 {"id":"T18-06","question":"Can temporal aspects be represented without hidden temporal metrics?","failure":"The representation secretly requires duration or rate."},
]
print(json.dumps(TESTS,indent=2))

## 13. Provisional findings packet

In [ ]:
FINDINGS={
 "notebook":18,
 "title":"Domain Projection and Primitive Hierarchy",
 "status":"conceptual_formalization_with_executable_examples",
 "findings":[
  {"id":"F18-01","statement":"Primitive status is indexed by declared domain."},
  {"id":"F18-02","statement":"Non-primitive status does not imply absence of meaning."},
  {"id":"F18-03","statement":"Added DoF may be operationalized as enlarged admissible relational capacity."},
  {"id":"F18-04","statement":"Projection requires an operator, target domain, and reference condition."},
  {"id":"F18-05","statement":"Resolution is generally many-to-one with respect to organization."},
  {"id":"F18-06","statement":"Projection path independence cannot be assumed."},
  {"id":"F18-07","statement":"Residue must be defined independently of physical energy, noise, or error."}
 ],
 "non_claims":[
  "(*|*) is not identified with a black hole.",
  "The notebook does not prove physical dimensions are generated by these illustrative maps.",
  "The example projection operators are not yet canonical RT operators.",
  "Combinatorial state growth alone is not treated as ontological proof."
 ],
 "next":{"notebook":19,"title":"Admissible Projection Algebra","goal":"Test closure, composition, equivalence, invariants, residue, and path dependence."}
}
print(json.dumps(FINDINGS,indent=2))

## Conclusion

Notebook 18 establishes the scaffold:

\[
oxed{	ext{Declare domain}	o	ext{declare status}	o	ext{declare projection}	o	ext{evaluate organization}	o	ext{record resolution and residue}}
\]

The arrows indicate ordinal dependency of inference, not temporal succession.

The next notebook should formalize admissible projection algebra rather than add another physical interpretation.